[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/purva-thakre/qce26-qrisp/blob/main/Session_2_Introduction_to_Jasp/BonusMaterial/PrimitivesInJax.ipynb)

# Primitives

## Setup

In [1]:
!pip install qrisp

In [2]:
import jax
import jax.numpy as jnp
import jax.lax as lax

## What Is a Primitive?

A jaxpr is built from primitive equations like `b:f32[] = sin a`. `sin` here is literally `lax.sin_p`, a specific `Primitive` object.

A primitive is a name plus a bundle of rules: how to run it on real data, trace it abstractly, and compile it. Here we just poke at existing ones before building our own.

Let's look at one directly:

In [3]:
print(f"sin primitive: {lax.sin_p}")
print(f"Type: {type(lax.sin_p).__name__}")

sin primitive: sin
Type: Primitive


The same holds for the other elementary ops JAX ships:

In [4]:
for name in ["add_p", "mul_p", "sin_p", "cos_p", "exp_p"]:
    if hasattr(lax, name):
        print(f"    {name}")

    add_p
    mul_p
    sin_p
    cos_p
    exp_p


Every elementary op we call through (`jnp` and then `sin`, `add`, `mul`, ...)  bottoms out in one of these `Primitive` objects. There's no hidden layer underneath.

## Tracing Through a Primitive

Let's consider a simple function that only returns the sine of a given object.

`sin` already has every rule it needs, so this section just watches those rules fire in turn: tracing and compiling, before we write them ourselves for a new primitive.

In [5]:
def f(x):
    return jnp.sin(x)


print(jax.make_jaxpr(f)(jnp.array(1.0)))

{ lambda ; a:f32[]. let b:f32[] = sin a in (b,) }


A single `sin` equation, same as we'd expect. 

When we trace the `jnp.sin` function, JAX calls the *abstract evaluation* rule of the `sin` primitive. In this case, it tells JAX that `sin` receives a scalar number and returns another scalar float32 number as output.

Now let's see it compiled:

In [6]:
x = jnp.array(1.0)

print("=== StableHLO for sin ===")
print(jax.jit(f).lower(x).as_text())

=== StableHLO for sin ===
module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>) -> (tensor<f32> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<f32>
    return %0 : tensor<f32>
  }
}



`stablehlo.sine` is XLA's spelling of the same `sin` primitive.

When we compile the `jnp.sin` function, JAX uses the *lowering* rule to translate the `sin` primitive into StableHLO. That is, the IR that `jit` then hands off to XLA to actually compile.

Observations:

- JAX uses many different rules: abstract evaluation for tracing, definition implementation for evaluation on concrete arrays, lowering for translating to the compiler's IR, batching for vectorization, differentiation for computing gradients, etc.

- `sin` already ships every rule it needs, so `jit` didn't have to do anything special to compile it.

- In Qrisp we mostly focus on abstract evaluation and definition implementation for the primitives we define in the Jasp module.

## Implementing a Custom Primitive

- Time to build one for real: `custom_primitive(x, y) = x**2 + y`, a small elementwise op that doesn't exist in JAX yet.
- The two rules that matter most are the ones that make a primitive exist and traceable at all: an implementation (`def_impl`) and an abstract-eval rule. We'll add those first, then a lowering rule so `jit` can compile it.
- JAX also supports differentiation and batching rules for a custom primitive: Qrisp's Jasp module doesn't define either for its own quantum primitives, so we'll skip them here too.

### Step 1: Definition Implementation

- `def_impl` is the eager-execution rule: what runs when you call `.bind` on concrete arrays, outside any tracing. Without it the primitive can't do anything at all.

In [7]:
from jax._src import core
import numpy as np

First, the primitive object itself: just a name registered with JAX, no rules attached yet:

In [8]:
# Create the primitive
custom_primitive_p = core.Primitive("custom_primitive")
custom_primitive_p.multiple_results = False  # single output, not a tuple

Now attach the eager-execution rule via `def_impl`:

In [9]:
@custom_primitive_p.def_impl
def custom_primitive_impl(x, y):
    return x**2 + y

Call it through `.bind`, the same entry point every primitive uses:

In [10]:
result = custom_primitive_p.bind(np.array([0.0, 1.0, 2.0]), np.array([1.0, 2.0, 3.0]))
print(f"Impl result: {result}")

Impl result: [1. 3. 7.]


- Notice `custom_primitive_impl` uses plain NumPy, not `jax.numpy`: this rule only ever sees concrete arrays, so it never needs to be traceable itself.

### Step 2: Abstract Evaluation

- `def_impl` alone can't trace the primitive. Tracing never touches real numbers, so JAX needs the output's shape/dtype in advance. That's `def_abstract_eval`.

- The `print` inside is there so you can see that abstractness directly, a couple of cells down.

In [11]:
from jax.core import ShapedArray


@custom_primitive_p.def_abstract_eval
def custom_primitive_abstract(x_aval, y_aval):
    print(f"x_aval: {x_aval}")
    return ShapedArray(
        x_aval.shape, x_aval.dtype
    )  # output shape/dtype = x's (assumes x, y already share a shape)

In [12]:
x = jnp.array([0.0, 1.0, 2.0])
y = jnp.array([1.0, 2.0, 3.0])

Wrap the `bind` call in a function so we have something to trace:

In [13]:
def f(x, y):
    return custom_primitive_p.bind(x, y)

Trace it:

In [14]:
print("Jaxpr with our primitive")
print(jax.make_jaxpr(f)(x, y))

Jaxpr with our primitive
x_aval: float32[3]
{ lambda ; a:f32[3] b:f32[3]. let c:f32[3] = custom_primitive a b in (c,) }


Observations:

- The `print` inside `custom_primitive_abstract` fired exactly once, during tracing, and showed a `ShapedArray`: shape and dtype only, no data. That's the abstractness `def_impl` alone couldn't give us.

- `custom_primitive` now sits in the jaxpr right alongside built-in primitives (`c:f32[3] = custom_primitive a b`), indistinguishable to the tracer from `sin` or `mul`.

### Step 3: JIT Lowering

- Abstract eval only fixes the output's shape; JAX still can't generate real code for `custom_primitive`. Watch it fail first:

In [15]:
try:
    jax.jit(f)(x, y)
except Exception as e:
    print(type(e).__name__)
    print(e)

NotImplementedError
MLIR translation rule for primitive 'custom_primitive' not found for platform cpu


- Exactly the missing piece: no MLIR translation rule exists yet for `custom_primitive`. `custom_primitive_lowering` builds one by hand, emitting the two StableHLO ops (`multiply`, `add`) that compute `x**2 + y`.

In [16]:
from jax._src.interpreters import mlir as jax_mlir
from jax._src.lib.mlir.dialects import hlo


def custom_primitive_lowering(ctx, x, y):
    squared = hlo.MulOp(x, x).result
    return [hlo.AddOp(squared, y).result]

In [17]:
jax_mlir.register_lowering(custom_primitive_p, custom_primitive_lowering)

With the rule registered, `jit` can finally run it:

In [18]:
jax.jit(f)(x, y)

Array([1., 3., 7.], dtype=float32)

And the compiled program itself:

In [19]:
print(jax.jit(f).lower(x, y).as_text())

module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3xf32>, %arg1: tensor<3xf32>) -> (tensor<3xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.multiply %arg0, %arg0 : tensor<3xf32>
    %1 = stablehlo.add %0, %arg1 : tensor<3xf32>
    return %1 : tensor<3xf32>
  }
}



- The compiled StableHLO matches the lowering rule line by line (`multiply`, `add`) so `jit` used exactly the rule we just registered, nothing generic.

- Qrisp's Jasp module builds custom primitives the exact same way: `QuantumPrimitive` objects with their own `def_impl`/`def_abstract_eval`, but none of them get an MLIR lowering rule registered through `jax.interpreters.mlir.register_lowering` like we just did above. There's nothing in `jax.jit`'s own lowering registry for a quantum gate.

- Instead, Jasp interprets a jaxpr containing quantum primitives directly: its own jaxpr-walking interpreter replays each equation in Python one at a time, dispatching quantum operations to a Python-level simulator, rather than compiling it through XLA. `jax.jit` still compiles the purely classical portions of a program the normal way. It's only quantum-touching code that takes this other path.

## What's Still Missing

- `custom_primitive` can now run eagerly, trace into a jaxpr, and compile under `jit`, but only those three rules exist. 

We can try `grad` and `vmap` on it (which are two JAX transformations):

In [20]:
try:
    jax.grad(lambda x: jnp.sum(custom_primitive_p.bind(x, y)))(x)
except Exception as e:
    print(type(e).__name__)
    print(e)

try:
    jax.vmap(lambda x, y: custom_primitive_p.bind(x, y))(
        jnp.stack([x, x]), jnp.stack([y, y])
    )
except Exception as e:
    print(type(e).__name__)
    print(e)

NotImplementedError
Differentiation rule for 'custom_primitive' not implemented


NotImplementedError
Batching rule for 'custom_primitive' not implemented


Observations:

- Both fail the same way: JAX tells us exactly which rule is missing (`Differentiation rule for 'custom_primitive' not implemented`, `Batching rule for 'custom_primitive' not implemented`) rather than silently doing the wrong thing.

- JAX lets us fill in both, a JVP (or `primitive_jvps` entry) for differentiation, a `batching.primitive_batchers` entry for `vmap`. Qrisp doesn't define either for its quantum primitives, so we stop here.

That covers the essential rules for a primitive: a name (`core.Primitive(...)`), an implementation (`def_impl`) for eager execution, an abstract-eval rule so it can be traced, and a lowering rule so `jit` can compile it. 

`grad` and `vmap` need their own rules too, which we deliberately left out above since Qrisp's Jasp module doesn't define them for its own quantum primitives either.